# Compile and install types from one Elm file

The native Elm frontend compiles a single file without project configuration, writes classic v3 IR and a task result, and installs a copy when requested. Rego checks the outputs; this proves type compilation, not native Elm function lowering.


In [ ]:
module Example exposing (Amount, Kind(..))


type alias Amount =
    { value : Int }


type Kind
    = Simple
    | Detailed Amount


## Compile the public Amount and Kind types


In [ ]:
morphir compile --input Example.elm --extension morphir-elm-native --package-name examples/single-file --json


In [ ]:
package step_1_test

import rego.v1

test_exit_code if {
    input.exitCode == 0
}

test_compile_reports_success if {
    input.stdoutJson["success"] == true
}

test_ir_format_version if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["formatVersion"] == 3
}

test_ir_is_library if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][0] == "Library"
}

test_package_name if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][1] == [["examples"],["single","file"]]
}

test_module_name if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][0] == [["example"]]
}

test_amount_type_name if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][0][0] == ["amount"]
}

test_amount_is_public if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][0][1]["access"] == "Public"
}

test_kind_type_name if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][1][0] == ["kind"]
}

test_kind_is_public if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][1][1]["access"] == "Public"
}

test_amount_is_alias if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][0][1]["value"]["value"][0] == "TypeAliasDefinition"
}

test_amount_record_fields if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][0][1]["value"]["value"][2] == ["Record",{},[{"name":["value"],"tpe":["Reference",{},[[["morphir"],["s","d","k"]],[["basics"]],["int"]],[]]}]]
}

test_kind_is_custom_type if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][1][1]["value"]["value"][0] == "CustomTypeDefinition"
}

test_simple_constructor if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][1][1]["value"]["value"][2]["value"][0] == [["simple"],[]]
}

test_detailed_constructor if {
    input.artifacts["ir"].kind == "json"
    input.artifacts["ir"].value["distribution"][3]["modules"][0][1]["value"]["types"][1][1]["value"]["value"][2]["value"][1][0] == ["detailed"]
}

test_task_ir_version if {
    input.artifacts["task"].kind == "json"
    input.artifacts["task"].value["ir"]["version"] == "v3"
}

test_no_implicit_install if {
    input.artifacts["installed"].kind == "missing"
}


## Install v3 IR while preserving the canonical output


In [ ]:
morphir compile --input Example.elm --extension morphir-elm-native --package-name examples/single-file --output installed --json


In [ ]:
package step_2_test

import rego.v1

test_exit_code if {
    input.exitCode == 0
}

test_compile_reports_success if {
    input.stdoutJson["success"] == true
}

test_canonical_ir_preserved if {
    input.artifacts["ir"].kind != "missing"
}

test_installed_ir_format_version if {
    input.artifacts["installed"].kind == "json"
    input.artifacts["installed"].value["formatVersion"] == 3
}

test_installed_package_name if {
    input.artifacts["installed"].kind == "json"
    input.artifacts["installed"].value["distribution"][1] == [["examples"],["single","file"]]
}

test_installed_module_name if {
    input.artifacts["installed"].kind == "json"
    input.artifacts["installed"].value["distribution"][3]["modules"][0][0] == [["example"]]
}
